In [3]:
## 셀 1: 저장된 그래프 + 곤란구간 리스트 불러오기

import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point, LineString

G_main = nx.read_graphml('grid_road_network_with_sidewalk.graphml')
difficult_segments = pd.read_csv('difficult_segments_final.csv')

print(f"그래프 노드 수: {G_main.number_of_nodes()}, 엣지 수: {G_main.number_of_edges()}")
print(f"곤란구간 링크 수: {len(difficult_segments)}")

그래프 노드 수: 7243, 엣지 수: 9930
곤란구간 링크 수: 27


In [9]:
## 셀 2: 원본 geometry 다시 불러와서 곤란구간 좌표 추출 (빠른 버전)

import pandas as pd
import geopandas as gpd

road_gdf = gpd.read_file('03._상세도로망_lev6.geojson')

# bbox를 read_file 단계에서 바로 적용 -> 전체 178만개 안 읽고 판교분당만 읽음
bbox = (127.05, 37.33, 127.16, 37.43)
osm_clip = gpd.read_file('gis_osm_roads_free_1.shp', bbox=bbox)
print(f"판교분당 클립 후: {len(osm_clip)}")

footway_types = ['footway', 'path', 'pedestrian', 'steps']
sidewalk = osm_clip[osm_clip['fclass'].isin(footway_types)].copy()

road_gdf_calc = road_gdf[['link_id', 'geometry']].copy()
sidewalk_calc = sidewalk[['osm_id', 'geometry']].copy()
sidewalk_calc['link_id'] = 'osm_' + sidewalk_calc['osm_id'].astype(str)
sidewalk_calc = sidewalk_calc[['link_id', 'geometry']]

all_links_gdf = pd.concat([road_gdf_calc, sidewalk_calc], ignore_index=True)
print(f"통합 링크 수: {len(all_links_gdf)}")

difficult_geom = all_links_gdf[all_links_gdf['link_id'].isin(difficult_segments['link_id'])].copy()
difficult_geom = difficult_geom.merge(difficult_segments[['link_id','link_hsi']], on='link_id', how='left')

difficult_geom['midpoint'] = difficult_geom.geometry.interpolate(0.5, normalized=True)
difficult_geom['lat'] = difficult_geom['midpoint'].y
difficult_geom['lon'] = difficult_geom['midpoint'].x

result = difficult_geom[['link_id', 'link_hsi', 'lat', 'lon']].sort_values('link_hsi', ascending=False).reset_index(drop=True)
result.to_csv('difficult_segments_coords.csv', index=False, encoding='utf-8-sig')
print(result)

판교분당 클립 후: 11625
통합 링크 수: 6592
      link_id  link_hsi        lat         lon
0   572543912  0.950600  37.384457  127.123495
1   571501063  0.942550  37.349848  127.108934
2   571551888  0.937454  37.396273  127.109447
3   571543551  0.921620  37.349194  127.108931
4   571541880  0.916116  37.383414  127.121581
5   571541909  0.904698  37.379209  127.114473
6   571500274  0.902470  37.380160  127.115803
7   571500307  0.899820  37.383337  127.121085
8   571541881  0.899228  37.383712  127.122480
9   572541693  0.896833  37.411721  127.128761
10  571541307  0.887483  37.383308  127.120522
11  571558228  0.869074  37.350036  127.110296
12  571556942  0.866003  37.396269  127.108900
13  571500575  0.865877  37.381547  127.116249
14  572544942  0.865114  37.410817  127.128759
15  572541686  0.859779  37.412272  127.128550
16  571556940  0.856231  37.396265  127.108113
17  572501277  0.839932  37.410082  127.128753
18  571542789  0.839272  37.339651  127.108969
19  572541689  0.829419  37.4

/tmp/ipykernel_9/4147273525.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  difficult_geom['midpoint'] = difficult_geom.geometry.interpolate(0.5, normalized=True)


In [10]:
## 셀 3: 곤란구간 27개 링크 시작점·중간점·끝점 좌표 전체 추출

difficult_geom['start_pt'] = difficult_geom.geometry.interpolate(0.0, normalized=True)
difficult_geom['end_pt'] = difficult_geom.geometry.interpolate(1.0, normalized=True)

difficult_geom['start_lat'] = difficult_geom['start_pt'].y
difficult_geom['start_lon'] = difficult_geom['start_pt'].x
difficult_geom['end_lat'] = difficult_geom['end_pt'].y
difficult_geom['end_lon'] = difficult_geom['end_pt'].x

# 링크 실제 길이(m)도 같이 확인 - 이게 짧으면 시작/끝이 거의 붙어있다는 뜻
difficult_geom_5186 = difficult_geom.to_crs(epsg=5186)
difficult_geom['length_m'] = difficult_geom_5186.geometry.length

result_full = difficult_geom[[
    'link_id', 'link_hsi', 'length_m',
    'start_lat', 'start_lon', 'lat', 'lon', 'end_lat', 'end_lon'
]].rename(columns={'lat':'mid_lat', 'lon':'mid_lon'}).sort_values('link_hsi', ascending=False).reset_index(drop=True)

result_full.to_csv('difficult_segments_coords_full.csv', index=False, encoding='utf-8-sig')
print(result_full)

      link_id  link_hsi    length_m  start_lat   start_lon    mid_lat  \
0   572543912  0.950600   73.420324  37.384230  127.123191  37.384457   
1   571501063  0.942550   40.626655  37.350031  127.108933  37.349848   
2   571551888  0.937454   44.657008  37.396275  127.109699  37.396273   
3   571543551  0.921620  104.387084  37.348724  127.108929  37.349194   
4   571541880  0.916116   44.764576  37.383565  127.121413  37.383414   
5   571541909  0.904698  132.597045  37.379606  127.115032  37.379209   
6   571500274  0.902470  183.863992  37.379606  127.115032  37.380160   
7   571500307  0.899820   77.103311  37.383108  127.120757  37.383337   
8   571541881  0.899228  169.797481  37.384230  127.123191  37.383712   
9   572541693  0.896833   80.378028  37.411359  127.128762  37.411721   
10  571541307  0.887483   60.866342  37.383108  127.120757  37.383308   
11  571558228  0.869074  241.515956  37.350031  127.108933  37.350036   
12  571556942  0.866003   52.157168  37.396271  127

/tmp/ipykernel_9/3981161875.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  difficult_geom['start_pt'] = difficult_geom.geometry.interpolate(0.0, normalized=True)
/tmp/ipykernel_9/3981161875.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  difficult_geom['end_pt'] = difficult_geom.geometry.interpolate(1.0, normalized=True)


In [12]:
## 목적지 좌표 정의 (12번 노트북에서 썼던 것과 동일)

destinations_v2 = {
    '야탑': (127.12873, 37.41131),
    '서현': (127.12397, 37.38474),
    '미금': (127.10891, 37.35016),
    '판교역': (127.10979, 37.39614),
}

In [13]:
## 셀 5: 목적지(역) 반경 200m 이내 곤란구간 제외

from shapely.geometry import Point
import geopandas as gpd

dest_points_5186 = {}
for name, (lon, lat) in destinations_v2.items():
    pt = gpd.GeoSeries([Point(lon, lat)], crs='EPSG:4326').to_crs(epsg=5186).iloc[0]
    dest_points_5186[name] = pt

# 곤란구간 중간점을 5186으로 변환해서 역들과의 최소거리 계산
difficult_geom_5186_check = difficult_geom.to_crs(epsg=5186)
difficult_geom_5186_check['mid_pt_5186'] = difficult_geom_5186_check.geometry.interpolate(0.5, normalized=True)

def min_dist_to_stations(pt):
    return min(pt.distance(dp) for dp in dest_points_5186.values())

difficult_geom_5186_check['dist_to_nearest_station'] = difficult_geom_5186_check['mid_pt_5186'].apply(min_dist_to_stations)

STATION_BUFFER = 200  # m
near_station = difficult_geom_5186_check[difficult_geom_5186_check['dist_to_nearest_station'] < STATION_BUFFER]
print(f"역 반경 {STATION_BUFFER}m 이내 제외 대상: {len(near_station)}개")
print(near_station[['link_id', 'link_hsi', 'dist_to_nearest_station']].sort_values('dist_to_nearest_station'))

difficult_segments_v3 = difficult_geom_5186_check[difficult_geom_5186_check['dist_to_nearest_station'] >= STATION_BUFFER]
print(f"\n최종 곤란구간(역세권 제외): {len(difficult_segments_v3)}개")

difficult_segments_v3[['link_id','link_hsi']].to_csv('difficult_segments_final_v3.csv', index=False, encoding='utf-8-sig')
print("저장 완료: difficult_segments_final_v3.csv")

역 반경 200m 이내 제외 대상: 16개
      link_id  link_hsi  dist_to_nearest_station
14  571551888  0.937454                33.781475
3   571501063  0.942550                34.740943
24  572541693  0.896833                45.718572
25  572543912  0.950600                52.609198
26  572544942  0.865114                54.740562
16  571556942  0.866003                80.092324
11  571543551  0.921620               107.201276
22  572541686  0.859779               107.263188
17  571558228  0.869074               123.551361
23  572541689  0.829419               126.571436
19  572501277  0.839932               136.269805
15  571556940  0.856231               149.140928
20  572541023  0.815673               161.207989
6   571541881  0.899228               174.016810
18  572542439  0.812505               174.826940
21  572541024  0.802618               194.060391

최종 곤란구간(역세권 제외): 11개
저장 완료: difficult_segments_final_v3.csv


In [14]:
## 셀 6: 최종 곤란구간(11개) 시작점·끝점 좌표 추출

final_11 = difficult_segments_v3.copy()

final_11['start_pt'] = final_11.geometry.interpolate(0.0, normalized=True)
final_11['end_pt'] = final_11.geometry.interpolate(1.0, normalized=True)

# 5186(미터좌표계)에서 뽑았으면 4326(위경도)으로 다시 변환 필요
final_11_4326 = final_11.to_crs(epsg=4326)
final_11_4326['start_pt_4326'] = final_11_4326.geometry.interpolate(0.0, normalized=True)
final_11_4326['end_pt_4326'] = final_11_4326.geometry.interpolate(1.0, normalized=True)

final_11_4326['start_lat'] = final_11_4326['start_pt_4326'].y
final_11_4326['start_lon'] = final_11_4326['start_pt_4326'].x
final_11_4326['end_lat'] = final_11_4326['end_pt_4326'].y
final_11_4326['end_lon'] = final_11_4326['end_pt_4326'].x

result_11 = final_11_4326[['link_id', 'link_hsi', 'start_lat', 'start_lon', 'end_lat', 'end_lon']].sort_values('link_hsi', ascending=False).reset_index(drop=True)

result_11.to_csv('difficult_segments_final11_coords.csv', index=False, encoding='utf-8-sig')
print(result_11)

      link_id  link_hsi  start_lat   start_lon    end_lat     end_lon
0   571541880  0.916116  37.383565  127.121413  37.383264  127.121748
1   571541909  0.904698  37.379606  127.115032  37.378811  127.113914
2   571500274  0.902470  37.379606  127.115032  37.380715  127.116574
3   571500307  0.899820  37.383108  127.120757  37.383565  127.121413
4   571541307  0.887483  37.383108  127.120757  37.383508  127.120287
5   571500575  0.865877  37.381792  127.116573  37.381301  127.115925
6   571542789  0.839272  37.339706  127.108972  37.339596  127.108966
7   571551868  0.822205  37.395993  127.107042  37.396264  127.107621
8   571542735  0.821341  37.382376  127.117396  37.381792  127.116573
9   571541906  0.817914  37.380715  127.116574  37.380935  127.116329
10  571549132  0.809193  37.393880  127.105411  37.394047  127.105610


/tmp/ipykernel_9/462854665.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  final_11_4326['start_pt_4326'] = final_11_4326.geometry.interpolate(0.0, normalized=True)
/tmp/ipykernel_9/462854665.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  final_11_4326['end_pt_4326'] = final_11_4326.geometry.interpolate(1.0, normalized=True)


In [16]:
## 셀 7 수정: adaptive_facility_total 직접 재생성 + 3축 집계

hsi_full = pd.read_csv('grid_HSI_selected_indicators_v2.csv')

# adaptive_facility_total 없으면 직접 만들기
if 'adaptive_facility_total' not in hsi_full.columns:
    hsi_full['adaptive_facility_total'] = (
        hsi_full['park_cnt'] + hsi_full['shade_cnt'] +
        hsi_full['cooling_shelter_cnt'] + hsi_full['smart_shelter_cnt']
    )

print(hsi_full.columns.tolist())

['grid_id', 'geometry', 'ADM_SIDO', 'ADM_SGG_CD', 'ADM_EMD_CD', 'ADM_EMD_NM', 'lon', 'lat', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'shade_cnt', 'cooling_shelter_cnt', 'daycare_cnt', 'smart_shelter_cnt', 'tree_road_len', 'NDVI_mean_5scene', 'LST_C_mean_5scene', 'LC1_100_시가화건조지역_ratio', 'GREEN_ratio_LC', 'senior_facility_total', 'IDW_mean', 'adaptive_facility_total']


In [17]:
## 셀 8: 10개 곤란구간 주변 격자 3축 지표 집계

final_10 = difficult_segments_v3[~difficult_segments_v3['link_id'].isin(['571542789'])].copy()
final_10_5186 = final_10.to_crs(epsg=5186) if final_10.crs != 'EPSG:5186' else final_10
final_10_5186['geometry'] = final_10_5186.geometry.buffer(12)

grid_5186 = gpd.GeoDataFrame(
    hsi_full,
    geometry=gpd.GeoSeries.from_wkt(hsi_full['geometry']) if hsi_full['geometry'].dtype == object else hsi_full['geometry'],
    crs='EPSG:4326'
).to_crs(epsg=5186)

overlay_10 = gpd.overlay(final_10_5186[['link_id','geometry']], grid_5186, how='intersection')

axis_summary = overlay_10.groupby('link_id').agg(
    시가화비율=('LC1_100_시가화건조지역_ratio', 'mean'),
    녹지비율=('GREEN_ratio_LC', 'mean'),
    NDVI=('NDVI_mean_5scene', 'mean'),
    적응시설_총합=('adaptive_facility_total', 'mean'),
).reset_index()

axis_summary = axis_summary.merge(final_10[['link_id','link_hsi']], on='link_id', how='left')
print(axis_summary.sort_values('link_hsi', ascending=False))

axis_summary.to_csv('difficult_segments_3axis_profile.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: difficult_segments_3axis_profile.csv")

      link_id     시가화비율      녹지비율      NDVI   적응시설_총합  link_hsi
4   571541880  1.000000  0.000000  0.356900  0.000000  0.916116
6   571541909  1.000000  0.000000  0.359200  0.000000  0.904698
0   571500274  1.000000  0.000000  0.267617  0.000000  0.902470
1   571500307  1.000000  0.000000  0.405950  0.500000  0.899820
3   571541307  1.000000  0.000000  0.415333  0.666667  0.887483
2   571500575  0.927600  0.071233  0.365500  0.000000  0.865877
8   571542789  0.893750  0.106250  0.350900  0.500000  0.839272
10  571551868  0.984950  0.000000  0.183350  0.000000  0.822205
7   571542735  0.799975  0.179900  0.290875  0.000000  0.821341
5   571541906  1.000000  0.000000  0.527000  0.000000  0.817914
9   571549132  1.000000  0.000000  0.257600  0.000000  0.809193

저장 완료: difficult_segments_3axis_profile.csv


In [18]:
## 셀 9: 오리역 제외 후 최종 10개로 확정

print("link_id dtype:", difficult_segments_v3['link_id'].dtype)

# dtype에 맞게 필터링 (문자열이든 숫자든 둘 다 대응)
final_10_correct = difficult_segments_v3[
    ~difficult_segments_v3['link_id'].astype(str).isin(['571542789'])
].copy()

print(f"제외 전: {len(difficult_segments_v3)}개")
print(f"오리역 제외 후: {len(final_10_correct)}개")

# 3축 프로파일도 다시 계산
final_10_correct_5186 = final_10_correct.to_crs(epsg=5186) if final_10_correct.crs != 'EPSG:5186' else final_10_correct
final_10_correct_5186['geometry'] = final_10_correct_5186.geometry.buffer(12)

overlay_10_correct = gpd.overlay(final_10_correct_5186[['link_id','geometry']], grid_5186, how='intersection')

axis_summary_final = overlay_10_correct.groupby('link_id').agg(
    시가화비율=('LC1_100_시가화건조지역_ratio', 'mean'),
    녹지비율=('GREEN_ratio_LC', 'mean'),
    NDVI=('NDVI_mean_5scene', 'mean'),
    적응시설_총합=('adaptive_facility_total', 'mean'),
).reset_index()

axis_summary_final = axis_summary_final.merge(final_10_correct[['link_id','link_hsi']], on='link_id', how='left')
print(axis_summary_final.sort_values('link_hsi', ascending=False))

axis_summary_final.to_csv('difficult_segments_3axis_profile_final10.csv', index=False, encoding='utf-8-sig')
final_10_correct.to_csv('difficult_segments_final10.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료")

link_id dtype: object
제외 전: 11개
오리역 제외 후: 10개
     link_id     시가화비율      녹지비율      NDVI   적응시설_총합  link_hsi
4  571541880  1.000000  0.000000  0.356900  0.000000  0.916116
6  571541909  1.000000  0.000000  0.359200  0.000000  0.904698
0  571500274  1.000000  0.000000  0.267617  0.000000  0.902470
1  571500307  1.000000  0.000000  0.405950  0.500000  0.899820
3  571541307  1.000000  0.000000  0.415333  0.666667  0.887483
2  571500575  0.927600  0.071233  0.365500  0.000000  0.865877
9  571551868  0.984950  0.000000  0.183350  0.000000  0.822205
7  571542735  0.799975  0.179900  0.290875  0.000000  0.821341
5  571541906  1.000000  0.000000  0.527000  0.000000  0.817914
8  571549132  1.000000  0.000000  0.257600  0.000000  0.809193

저장 완료
